In [2]:
# ==============================================================================
# WEEK 2: Baseline Model and Data Loading Pipeline (PyTorch)
# ==============================================================================

# 1. Mount Google Drive to access the zipped dataset
from google.colab import drive
drive.mount('/content/drive')

# 2. Extract the dataset from Google Drive to Colab's local directory
# unzips all 35,000 images
!unzip -q /content/drive/MyDrive/DL_Project/fer2013.zip -d ./data

# 3. Import required PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set device (use GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 4. Data Preprocessing Setup
# FER-2013 contains 48x48 grayscale images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), # Ensure 1 channel (grayscale)
    transforms.Resize((48, 48)),                 # Standardize size to 48x48
    transforms.ToTensor(),                       # Convert images to PyTorch tensors
    transforms.Normalize((0.5,), (0.5,))         # Normalize pixel values to [-1, 1]
])

# Define paths to the extracted data folders inside Colab
train_dir = './data/train'
test_dir = './data/test'

# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

# Create DataLoaders to fetch data in batches of 64
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Successfully loaded {len(train_dataset)} training images and {len(test_dataset)} test images.")

# 5. Baseline Model Definition (Simple CNN)
class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        # First convolutional layer (1 input channel -> 16 output channels)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces size by half (48x48 -> 24x24)

        # Second convolutional layer (16 channels -> 32 output channels)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        # Second pool reduces size by half again (24x24 -> 12x12)

        # Fully connected layers (dense layers)
        self.fc1 = nn.Linear(32 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 7) # 7 target classes for basic emotions

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # Flatten the tensor for the dense layer
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and move it to GPU/CPU
model = BaselineCNN().to(device)
print("\n--- Model Architecture ---")
print(model)

# 6. Training Configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 7. Training Loop to establish baseline performance
epochs = 2 # 2 epochs is enough to quickly show that the pipeline works
print("\n--- Starting Training Loop ---")

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Tracking statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}, Accuracy: {epoch_acc:.2f}%")

print("\nBaseline model trained successfully! Baseline established.")

Mounted at /content/drive
Using device: cuda
Successfully loaded 28709 training images and 7178 test images.

--- Model Architecture ---
BaselineCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4608, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)

--- Starting Training Loop ---
Epoch 1/2 - Loss: 1.5976, Accuracy: 37.54%
Epoch 2/2 - Loss: 1.3820, Accuracy: 47.34%

Baseline model trained successfully! Baseline established.
